In [ ]:
from snowflake.ml.registry import Registry 
from snowflake.snowpark.context import get_active_session
import pandas as pd


session = get_active_session()


reg = Registry(
    session=session,
    database_name="DOCS_DB",   # e.g. "ML_DB"
    schema_name="MAIN", # e.g. "MODEL_REGISTRY"
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline


MODEL_NAME = "OpenMed/OpenMed-NER-DiseaseDetect-SuperClinical-434M"


class BatchedNERModel(custom_model.CustomModel):
    """Custom model wrapper that enables efficient GPU batching for HuggingFace NER pipeline."""
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        self.pipeline = None
        self.batch_size = 32
    
    @custom_model.inference_api
    def predict(self, inputs: pd.DataFrame) -> pd.DataFrame:
        import torch
        
        if self.pipeline is None:
            model_dir = self.context.path("model_artifacts")
            
            device = 0 if torch.cuda.is_available() else -1
            
            tokenizer = AutoTokenizer.from_pretrained(model_dir)
            model = AutoModelForTokenClassification.from_pretrained(model_dir)
            
            self.pipeline = pipeline(
                task="token-classification",
                model=model,
                tokenizer=tokenizer,
                aggregation_strategy="simple",
                device=device,
                batch_size=self.batch_size,
            )
        
        texts = inputs["inputs"].tolist()
        results = self.pipeline(texts)
        
        if isinstance(results[0], dict):
            results = [results]
        
        return pd.DataFrame({"output": [str(r) for r in results]})



reg = Registry(
    session=session,
    database_name="DOCS_DB",
    schema_name="MAIN",
)

print("Downloading model artifacts locally...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

local_model_dir = "/tmp/openmed_ner_model"
os.makedirs(local_model_dir, exist_ok=True)
tokenizer.save_pretrained(local_model_dir)
model.save_pretrained(local_model_dir)
print(f"Model saved to {local_model_dir}")

batched_model = BatchedNERModel(
    context=custom_model.ModelContext(
        artifacts={"model_artifacts": local_model_dir}
    )
)

In [ ]:

sample_input = pd.DataFrame({
    "inputs": [
        "Ms. Johnson is a gastroenterology patient seen on 2025-09-01. She denies chest pain but endorses night sweats in the setting of generalized anxiety disorder.",
        "Mr. Patel is a oncology patient seen on 2025-09-12. She denies chest pain but endorses night sweats in the setting of type 2 diabetes mellitus."
    ]
})




print("Logging model to registry...")
ner_model = reg.log_model(
    batched_model,
    model_name="ner_openmed_batched",
    version_name="v3",
    sample_input_data=sample_input,
    pip_requirements=["sentence-transformers", "torch", "transformers"],
    options={"cuda_version": "12.3"}
)





In [ ]:
ner_model.create_service(
    service_name="ner_openmed_batched_v3_svc",
    service_compute_pool="GPU_ML_M_POOL",
    ingress_enabled=True,
    gpu_requests="4",
    max_instances=4,
    num_workers=4,
    max_batch_rows=64,
)


In [ ]:
# Test inference
model = reg.get_model("ner_openmed_batched")
mv = model.version("v3")

df = session.sql('SELECT TEXT AS "inputs" FROM DOCS_DB.MAIN.SYNTH_DISEASE_NOTES LIMIT 100').to_pandas()

df_out = mv.run(
    df,
    function_name="predict",
    service_name="ner_openmed_batched_v3_svc",
)
df_out

In [ ]:
--1,000,000 rows: 9m35s !!
with cte as (
  select text as "inputs"
  from synth_disease_notes
) select docs_db.main.ner_openmed_batched_v3_svc!predict("inputs") AS classification_output
from cte
;